# Cell-Type Annotation

**REQUIRED DAY 2**

## Load your checkpoint

Fresh kernel -- loading back the `adata` saved at the end of [07_dimensionality_reduction_and_clustering.ipynb](07_dimensionality_reduction_and_clustering.ipynb) (PCA, neighbors, UMAP, `leiden` clusters already computed).

In [ ]:
import scanpy as sc

adata = sc.read_h5ad("results/checkpoint_07_clustered.h5ad")
adata


## From clusters to cell types

Clustering (previous notebook) gives you groups of transcriptionally similar cells. It does not tell you what those groups *are* — that's a separate, biological step: matching each cluster's characteristic genes against known marker genes for expected cell types.

Today's sample is PBMCs (peripheral blood mononuclear cells), so the expected cast is roughly: T cells, B cells, NK cells, and monocytes.

| Marker gene | Cell type |
| --- | --- |
| CD3D, CD3E | T cells (all) |
| CD8A | Cytotoxic T cells |
| MS4A1, CD19 | B cells |
| NKG7, GNLY | NK cells |
| LYZ, CD14 | Classical monocytes |
| FCGR3A | Non-classical monocytes |
| PPBP | Platelets |

## Annotation backed by a statistic, not a glance

It's tempting to color the UMAP by one marker gene, squint, and declare a cluster's identity. Do this instead — find each cluster's actual top differentially expressed genes and check them against the marker panel:

In [ ]:
sc.tl.rank_genes_groups(adata, groupby="leiden", method="wilcoxon")
sc.pl.rank_genes_groups(adata, n_genes=10, sharey=False)

In [ ]:
PBMC_MARKERS = ["CD3D", "CD3E", "CD8A", "MS4A1", "CD19", "NKG7", "GNLY", "LYZ", "CD14", "FCGR3A", "PPBP"]
sc.pl.dotplot(adata, PBMC_MARKERS, groupby="leiden")

`rank_genes_groups` runs a statistical test (Wilcoxon rank-sum, by default) for every gene, per cluster, against the rest of the data — you get an actual ranked, tested list, not an impression. This is Agent-B checklist item 17: is each cell-type annotation backed by a statistical marker test, not just a colored UMAP that "looks right"?

Once you're satisfied, label the clusters:

In [ ]:
adata.obs["cell_type"] = adata.obs["leiden"].map({
    "0": "CD4 T cells",
    "1": "CD14+ Monocytes",
    # ... fill in based on what rank_genes_groups actually shows you
})
sc.pl.umap(adata, color="cell_type")

## Why the full genome index mattered (the payoff)

Back in [03_raw_data_fastq_to_counts.md](03_raw_data_fastq_to_counts.md), the plan was to pre-build a **full** genome index rather than restrict it to one chromosome, because the marker panel above spans chromosomes 1, 2, 4, 5, 11, 12, 16, and 19 (GRCh37 coordinates). If the index had been restricted to save build time, some of these markers would have been silently unmappable, and you would have seen fewer expected cell types today for a reason that had nothing to do with the actual biology of this sample.

## Agent-assisted annotation, done right

> Weak: "Label these clusters."
>
> Strong: "For each Leiden cluster, run `rank_genes_groups` and show me the top 10 genes by test statistic, then propose a cell-type label only where at least one gene from the marker panel below appears in that cluster's top genes — flag any cluster where it doesn't, rather than guessing."

## Practice: the full checklist, end to end

This is today's capstone validation exercise. Open a fresh Agent B session and run the **entire extended 18-item checklist** from [02_agent_assisted_scrna_workflow.md](02_agent_assisted_scrna_workflow.md) against your complete pipeline — QC through annotation — not just the items specific to this notebook. Write down, for each item, the PASS/WARNING/FAIL verdict, the evidence, and the smallest correction if it's not a clean PASS.

## Further reading

- [Single-cell best practices — Annotation](https://www.sc-best-practices.org/cellular_structure/annotation.html)
- [scanpy: `rank_genes_groups` documentation](https://scanpy.readthedocs.io/en/stable/generated/scanpy.tl.rank_genes_groups.html)